In [59]:
import pandas as pd
import re

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer


from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report , accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import LabelEncoder

import joblib





In [39]:
df = pd.read_csv("municipal_training_set_1100.csv")
df.head()

,report_id,resident_id,resident_name,resident_email,resident_phone,issue_type,issue_description,issue_status,report_datetime,location_street_address,...,photo_url,municipal_department,resolution_datetime,resolution_notes,is_anonymous,priority_level,priority_score,severity_score,area_importance,citizen_reports_count
0,RPT-354769,RES-1028,Jack Hernandez,jack.hernandez28@example.com,+1-758-555-1049,noise,Unauthorized late-night party with amplified s...,resolved,2025-10-21T19:29:14.031832,3422 Sunset Blvd,...,NaN,Public Safety,2025-10-30T19:29:14.031832,Resolved by field unit.,True,low,3.08,2,7,3
1,RPT-928980,RES-6964,NaN,NaN,NaN,pothole,Deep pothole near the bus stop making it dange...,in-progress,2023-07-10T08:15:00,1200 W 87th St,...,https://cityreports.com/photos/pothole1200.jpg,Road Maintenance,NaN,NaN,True,medium,6.30,9,7,10
2,RPT-337675,RES-2013,William Gonzalez,william.gonzalez@example.com,+1-362-555-3105,noise,Construction crew is starting their work at 5 ...,in-progress,2025-07-11T00:00:00,9390 Cedar Ln,...,NaN,Public Safety,NaN,NaN,False,medium,6.02,5,7,32
3,RPT-528487,RES-5089,Patricia Davis,patricia.davis104@mail.com,+1-380-555-5810,water_leakage,URGENT: A significant water leak near the scho...,submitted,2025-06-15T00:00:00,1254 Oak Rd,...,https://civicphotos.gov.us/water_leakage-RPT-5...,Water Department,NaN,NaN,False,high,7.72,7,9,37
4,RPT-609991,RES-1044,Susan Williams,susan.williams26@example.com,+1-281-555-1496,pothole,Can someone please fix a small pothole that is...,submitted,2025-05-12T19:27:31.182002,8226 Cedar Dr,...,NaN,Roads & Highways,NaN,NaN,True,low,3.46,4,4,11


In [40]:
df.tail()

,report_id,resident_id,resident_name,resident_email,resident_phone,issue_type,issue_description,issue_status,report_datetime,location_street_address,...,photo_url,municipal_department,resolution_datetime,resolution_notes,is_anonymous,priority_level,priority_score,severity_score,area_importance,citizen_reports_count
1095,RPT-757189,RES-2066,David Hernandez,david.hernandez@example.com,+1-365-555-7058,pothole,Small but sharp pothole that is definitely goi...,submitted,2025-12-17T00:00:00,5191 Cedar Ln,...,NaN,Roads & Highways,NaN,NaN,False,medium,6.20,5,10,20
1096,RPT-514367,RES-1021,Elizabeth Davis,elizabeth.davis@example.com,+1-747-555-4335,illegal_parking,"Car parked in front of the driveway, blocking ...",resolved,2025-10-28T00:00:00,3210 Park Blvd,...,https://civicphotos.gov.us/illegal_parking-RPT...,Traffic Department,2025-10-30T00:00:00,Issue addressed by field team and marked closed.,False,low,3.18,3,4,13
1097,RPT-447166,RES-5345,James Martinez,james.martinez727@mail.com,+1-240-555-8912,pothole,EMERGENCY: THE POTHOLE ON THE MAIN ROAD IS GET...,submitted,2025-12-24T00:00:00,4866 Lake Blvd,...,https://civicphotos.gov.us/pothole-RPT-447166.jpg,Roads & Highways,NaN,NaN,False,critical,9.52,10,9,47
1098,RPT-797475,RES-5396,Richard Rodriguez,richard.rodriguez815@mail.com,+1-944-555-3695,water_leakage,EMERGENCY: A SIGNIFICANT WATER LEAK NEAR THE S...,submitted,2026-04-28T00:00:00,8165 Broadway Blvd,...,https://civicphotos.gov.us/water_leakage-RPT-7...,Water Department,NaN,NaN,False,critical,9.48,9,10,48
1099,RPT-616146,RES-5161,Michael Martinez,michael.martinez212@mail.com,+1-300-555-5407,illegal_parking,URGENT: Multiple vehicles are double-parked al...,submitted,2025-08-06T00:00:00,4659 Broadway Rd,...,https://civicphotos.gov.us/illegal_parking-RPT...,Traffic Department,NaN,NaN,False,high,7.42,7,9,32


In [41]:
df = df.dropna(subset=["issue_description"]) # as we need the column most important to perform TF-IDF

df = df.dropna(subset=["issue_type"])  # issue type also given 

### Select required columns

In [42]:
df = df[["issue_description", "issue_type"]]

### Clean data

In [43]:
df = df.dropna()
df.columns = ["text", "category"]

In [44]:
df['category'] = df['category'].replace({
    'water-leak': 'water_leakage',
    'illegal-dumping': 'illegal_dumping'
})

### Text preprocessing

In [45]:

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()  # removes extra spaces
    return text

### NOW as we are planning to make a full pipe line so it becomes easy for us later on 

In [46]:


category_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=clean_text)),
    ('model', LogisticRegression(
        max_iter=1000,        # needed for convergence
        #multi_class='auto',   # handles multiple issue_types
        class_weight='balanced'  # handles any imbalance
    ))
])

### test and train

In [47]:


X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["category"], test_size=0.2 , random_state=42
)

### TRAINING our model to predict the categories 

In [48]:
category_pipeline.fit(X_train, y_train)

,steps,"[('tfidf', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,<function cle...x764db84058a0>
,tokenizer,None


### Evaluationn

In [49]:
    

preds = category_pipeline.predict(X_test)
print(classification_report(y_test, preds))
print("Accuracy:", accuracy_score(y_test, preds))

                 precision    recall  f1-score   support

       drainage       0.97      1.00      0.98        29
        garbage       0.97      0.97      0.97        31
       graffiti       1.00      1.00      1.00         4
illegal_dumping       1.00      1.00      1.00         3
illegal_parking       1.00      1.00      1.00        29
          noise       0.96      1.00      0.98        22
          other       1.00      0.86      0.92         7
        pothole       1.00      0.97      0.98        33
    streetlight       0.96      1.00      0.98        26
  water_leakage       1.00      0.97      0.99        36

       accuracy                           0.98       220
      macro avg       0.99      0.98      0.98       220
   weighted avg       0.98      0.98      0.98       220

Accuracy: 0.9818181818181818


### TESTING OUR MODEL

In [50]:
# text = ["large pothole causing traffic issues on main road"] ## suppose we gave this description

# prediction = category_pipeline.predict(text)

# print(prediction)

In [51]:
joblib.dump(category_pipeline, 'category_model.pkl')
print("Model saved!")

Model saved!


In [52]:
loaded_model = joblib.load('category_model.pkl')

test_desc = ["garbage not collected since three days"]
prediction = loaded_model.predict(test_desc)
print("Predicted category:", prediction[0])

Predicted category: garbage


and yes it correctly predicted the category 

## AS we have categoriezed the complaints into categories and now we are giving PRIORITY to our complaints


## PRIORITY MODEL

In [62]:
df_priority = pd.read_csv('municipal_training_set_1100.csv')

In [63]:
df_priority['issue_type'] = df_priority['issue_type'].replace({
    'water-leak': 'water_leakage',
    'illegal-dumping': 'illegal_dumping'
})

In [68]:
df_priority.head()


,report_id,resident_id,resident_name,resident_email,resident_phone,issue_type,issue_description,issue_status,report_datetime,location_street_address,...,photo_url,municipal_department,resolution_datetime,resolution_notes,is_anonymous,priority_level,priority_score,severity_score,area_importance,citizen_reports_count
0,RPT-354769,RES-1028,Jack Hernandez,jack.hernandez28@example.com,+1-758-555-1049,noise,Unauthorized late-night party with amplified s...,resolved,2025-10-21T19:29:14.031832,3422 Sunset Blvd,...,NaN,Public Safety,2025-10-30T19:29:14.031832,Resolved by field unit.,True,low,3.08,2,7,3
1,RPT-928980,RES-6964,NaN,NaN,NaN,pothole,Deep pothole near the bus stop making it dange...,in-progress,2023-07-10T08:15:00,1200 W 87th St,...,https://cityreports.com/photos/pothole1200.jpg,Road Maintenance,NaN,NaN,True,medium,6.30,9,7,10
2,RPT-337675,RES-2013,William Gonzalez,william.gonzalez@example.com,+1-362-555-3105,noise,Construction crew is starting their work at 5 ...,in-progress,2025-07-11T00:00:00,9390 Cedar Ln,...,NaN,Public Safety,NaN,NaN,False,medium,6.02,5,7,32
3,RPT-528487,RES-5089,Patricia Davis,patricia.davis104@mail.com,+1-380-555-5810,water_leakage,URGENT: A significant water leak near the scho...,submitted,2025-06-15T00:00:00,1254 Oak Rd,...,https://civicphotos.gov.us/water_leakage-RPT-5...,Water Department,NaN,NaN,False,high,7.72,7,9,37
4,RPT-609991,RES-1044,Susan Williams,susan.williams26@example.com,+1-281-555-1496,pothole,Can someone please fix a small pothole that is...,submitted,2025-05-12T19:27:31.182002,8226 Cedar Dr,...,NaN,Roads & Highways,NaN,NaN,True,low,3.46,4,4,11


In [70]:
df_priority['issue_type'] = LabelEncoder().fit_transform(df_priority['issue_type'])

In [67]:
features = ['severity_score', 'area_importance', 
            'citizen_reports_count', 'issue_type']
target = 'priority_level'

In [72]:
X = df_priority[features]
y = df_priority[target]

In [73]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [74]:
rf_model = RandomForestClassifier(
    n_estimators=100,        # 100 trees
    class_weight='balanced', # handles class imbalance
    random_state=42          # consistent results
)
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [75]:
y_pred = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.9727272727272728

              precision    recall  f1-score   support

    critical       1.00      1.00      1.00        41
        high       1.00      0.97      0.98        63
         low       0.97      0.93      0.95        42
      medium       0.94      0.99      0.96        74

    accuracy                           0.97       220
   macro avg       0.98      0.97      0.97       220
weighted avg       0.97      0.97      0.97       220



In [76]:
joblib.dump(rf_model, 'priority_model.pkl')
print("Model saved!")

Model saved!


### Now we are testing what we have build is our trained model giving correct output or not 

In [56]:
# text = ["garbage not collected near street"]

# print(category_pipeline.predict(text))
# print(priority_pipeline.predict(text))

In [1]:
pip show scikit-learn


Name: scikit-learn
Version: 1.7.2
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License-Expression: BSD-3-Clause
Location: /home/wajahat/anaconda3/lib/python3.13/site-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: imbalanced-learn
Note: you may need to restart the kernel to use updated packages.
